# Built-in BasicTS Model Benchmark

This section benchmarks several built-in BasicTS forecasting models on the ETTh1 dataset using the same shared settings and training procedure, then ranks them by performance.


In [1]:
# Project setup: ensure repo root and src are on the path
import os
import sys
from pathlib import Path
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
os.chdir(ROOT)
src_path = ROOT / "src"
if str(src_path) not in sys.path:
	sys.path.insert(0, str(src_path))


In [ ]:
# Imports and shared benchmark settings
import json
import importlib
import traceback
import pkgutil
from pathlib import Path
from math import sqrt
import pandas as pd

import torch

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode

# Shared dataset / training settings for the benchmark
DATASET_NAME = "ETTh1"
INPUT_LEN = 96
OUTPUT_LEN = 12
NUM_FEATURES = 7
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

# Shared BasicTS config fields used for every model
SHARED_CONFIG = {
	"dataset_name": DATASET_NAME,
	"input_len": INPUT_LEN,
	"dataset_params": {
		"input_len": INPUT_LEN,
		"output_len": OUTPUT_LEN,
		"use_timestamps": False,
		"memmap": False,
	},
	"use_timestamps": False,
	"batch_size": BATCH_SIZE,
	"num_epochs": NUM_EPOCHS,
	"scaler": ZScoreScaler,
	"norm_each_channel": True,
	"rescale": False,
	"metrics": ["MAE", "MSE", "RMSE", "MAPE", "WAPE"],
	"optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
	"gpus": None,
	"train_data_num_workers": 0,
	"val_data_num_workers": 0,
	"test_data_num_workers": 0,
	"save_results": True,
}

# Auto-discover model modules inside basicts.models that expose a Config and a forecasting class
import basicts.models as _models_pkg

def discover_models():
	candidates = []
	for finder, name, ispkg in pkgutil.iter_modules(_models_pkg.__path__):
		try:
			mod = importlib.import_module(f"basicts.models.{name}")
		except Exception:
			continue
		has_config = any(attr.endswith("Config") for attr in dir(mod))
		has_forecast = any("Forecast" in attr for attr in dir(mod))
		has_modelname = hasattr(mod, name)
		if has_config and (has_forecast or has_modelname):
			candidates.append(name)
	return sorted(candidates)

discovered = discover_models()
print("Discovered model modules:", discovered)

# User-requested selection: 3 simple and 3 advanced models (in priority order)
DESIRED_SIMPLE = ["DLinear", "NLinear", "MTSMixer"]
DESIRED_ADVANCED = ["TimesNet", "iTransformer", "PatchTST"]

# Build MODEL_REGISTRY by taking available models from the desired lists in order
MODEL_REGISTRY = []
for name in (DESIRED_SIMPLE + DESIRED_ADVANCED):
	if name in discovered:
		MODEL_REGISTRY.append(name)

print("Using MODEL_REGISTRY (simple first, then advanced):", MODEL_REGISTRY)

# Where to save benchmark results
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


Discovered model modules: ['Autoformer', 'DLinear', 'FITS', 'FiLM', 'FreTS', 'HI', 'Informer', 'Koopa', 'Leddam', 'LightTS', 'MTSMixer', 'NLinear', 'NonstationaryTransformer', 'PatchTST', 'SOFTS', 'STID', 'SegRNN', 'SparseTSF', 'StemGNN', 'TiDE', 'TimeKAN', 'TimeMixer', 'TimeXer', 'Timer', 'TimesNet', 'iTransformer']


In [ ]:
# Benchmark runner: imports model classes/configs, trains and evaluates, and collects metrics.
import os
from pathlib import Path
from glob import glob

def find_attr_case(module, candidates):
	for name in candidates:
		if hasattr(module, name):
			return getattr(module, name)
	return None

available_models = []
failed_models = []
results = []

for model_name in MODEL_REGISTRY:
	try:
		mod = importlib.import_module(f"basicts.models.{model_name}")
	except Exception as e:
		failed_models.append({"model": model_name, "error": f"import error: {e}"})
		continue

	# Heuristics to find a forecasting wrapper or a model class
	# Prefer classes mentioning 'Forecast' in their name, else fallback to class named like model_name
	model_cls = None
	for attr in dir(mod):
		if "Forecast" in attr:
			model_cls = getattr(mod, attr)
			break
	if model_cls is None and hasattr(mod, model_name):
		model_cls = getattr(mod, model_name)

	# Find a config dataclass
	config_cls = None
	if hasattr(mod, f"{model_name}Config"):
		config_cls = getattr(mod, f"{model_name}Config")
	else:
		# fallback: any attribute that endswith Config
		for attr in dir(mod):
			if attr.endswith("Config"):
				config_cls = getattr(mod, attr)
				break

	if model_cls is None or config_cls is None:
		failed_models.append({"model": model_name, "error": "missing model class or config in module"})
		continue

	print(f"Benchmarking {model_name}...")

	# instantiate model config with shared required fields; configs have defaults for optional args
	try:
		# Some config classes accept num_features, some don't; pass what is common
		model_config = config_cls(input_len=INPUT_LEN, output_len=OUTPUT_LEN, num_features=NUM_FEATURES)
	except TypeError:
		# fallback without num_features
		model_config = config_cls(input_len=INPUT_LEN, output_len=OUTPUT_LEN)

	ckpt_dir = Path(f"checkpoints/benchmark/{model_name}/{DATASET_NAME}_{INPUT_LEN}_{OUTPUT_LEN}")

	cfg = BasicTSForecastingConfig(
		model=model_cls,
		model_config=model_config,
		taskflow=BasicTSForecastingTaskFlow(),
		ckpt_save_dir=str(ckpt_dir),
		**SHARED_CONFIG,
	)

	# Train and evaluate, catching any errors so the loop continues
	try:
		BasicTSLauncher.launch_training(cfg)
	except Exception as e:
		tb = traceback.format_exc()
		failed_models.append({"model": model_name, "error": f"train error: {e}\n{tb}"})
		continue

	try:
		# Passing None lets the runner pick the best checkpoint according to target metric
		BasicTSLauncher.launch_evaluation(cfg, None)
	except Exception as e:
		tb = traceback.format_exc()
		failed_models.append({"model": model_name, "error": f"eval error: {e}\n{tb}"})
		# still try to find any saved metrics

	# Find the most recent test_metrics.json under the ckpt_dir tree
	metrics_files = list(Path(ckpt_dir).rglob("test_metrics.json"))
	if not metrics_files:
		# no metrics saved
		failed_models.append({"model": model_name, "error": "no test_metrics.json found after eval"})
		continue

	metrics_file = max(metrics_files, key=lambda p: p.stat().st_mtime)
	try:
		with open(metrics_file, "r") as f:
			metrics = json.load(f)
	except Exception as e:
		failed_models.append({"model": model_name, "error": f"failed reading metrics: {e}"})
		continue

	overall = metrics.get("overall", {})

	# Normalize / compute missing metrics
	mae = overall.get("MAE")
	mse = overall.get("MSE")
	rmse = overall.get("RMSE") if overall.get("RMSE") is not None else (sqrt(mse) if mse is not None else None)
	mape = overall.get("MAPE")
	wape = overall.get("WAPE")

	results.append({
		"model": model_name,
		"MAE": mae,
		"MSE": mse,
		"RMSE": rmse,
		"MAPE": mape,
		"WAPE": wape,
		"metrics_file": str(metrics_file),
		"ckpt_dir": str(ckpt_dir),
	})

# Save results
df = pd.DataFrame(results)
if not df.empty:
	df_sorted = df.sort_values(by=["MAE", "MSE"], na_position="last")
	df_sorted.to_csv(RESULTS_DIR / "model_benchmark_results.csv", index=False)
	df_sorted.to_json(RESULTS_DIR / "model_benchmark_results.json", orient="records", indent=2)

# Save failed models
fm_df = pd.DataFrame(failed_models)
if not fm_df.empty:
	fm_df.to_csv(RESULTS_DIR / "model_benchmark_failed.csv", index=False)

print("Benchmark finished. Results saved to results/")


2026-06-16 09:22:49,283 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-16 09:22:49,296 - BasicTS - INFO - Building model.
2026-06-16 09:22:49,300 - BasicTS - INFO - Set ckpt save dir: "checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a36e17a575e4f938c2d8d175d2888"
2026-06-16 09:22:49,302 - BasicTS-training - INFO - Initializing training.
2026-06-16 09:22:49,302 - BasicTS-training - INFO - Building train data loader.


Benchmarking DLinear...


2026-06-16 09:22:51,138 - BasicTS-training - INFO - Set optim: Adam
2026-06-16 09:22:51,139 - BasicTS-training - INFO - Building val data loader.
2026-06-16 09:22:51,147 - BasicTS-training - INFO - Building test data loader.
2026-06-16 09:22:51,149 - BasicTS-training - INFO - Total parameters: 2328
2026-06-16 09:22:51,150 - BasicTS-training - INFO - Trainable parameters: 2328
2026-06-16 09:22:51,150 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [00:00<00:00, 279.85it/s]
2026-06-16 09:22:52,111 - BasicTS-training - INFO - Result <train>: [train/time: 0.96 (s), train/loss: 0.3860, train/MAE: 0.3860, train/MSE: 0.3241, train/RMSE: 0.5693, train/MAPE: 5.0397, train/WAPE: 0.7198]
2026-06-16 09:22:52,114 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 489.99it/s]
2026-06-16 09:22:52,295 - BasicTS-training - INFO - Result <val>: [val/time: 0.18 (s), val/loss: 0.3935, val/MAE: 0.3935, val/MSE: 0.3471, val/RMSE: 0.5891, val/MAPE: 5.3301, val

Benchmarking NLinear...


100%|██████████| 267/267 [00:00<00:00, 418.38it/s]
2026-06-16 09:22:58,402 - BasicTS-training - INFO - Result <train>: [train/time: 0.64 (s), train/loss: 0.3831, train/MAE: 0.3831, train/MSE: 0.3176, train/RMSE: 0.5636, train/MAPE: 5.3838, train/WAPE: 0.7289]
2026-06-16 09:22:58,405 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 689.03it/s]
2026-06-16 09:22:58,534 - BasicTS-training - INFO - Result <val>: [val/time: 0.13 (s), val/loss: 0.3935, val/MAE: 0.3935, val/MSE: 0.3524, val/RMSE: 0.5936, val/MAPE: 5.8268, val/WAPE: 0.6467]
2026-06-16 09:22:58,538 - BasicTS-training - INFO - Checkpoint checkpoints\benchmark\NLinear\ETTh1_96_12\5484d362080ec2be957991908b835bf3\NLinear_best_val_MAE.pt saved
100%|██████████| 87/87 [00:00<00:00, 697.79it/s]
2026-06-16 09:22:58,665 - BasicTS-training - INFO - Result <test>: [test/time: 0.13 (s), test/loss: 0.3491, test/MAE: 0.3491, test/MSE: 0.2999, test/RMSE: 0.5477, test/MAPE: 8.8855, test/WAPE: 0.5888]
2026-06-16

...

Benchmarking iTransformer...


2026-06-16 09:23:23,838 - BasicTS-training - INFO - Initializing training.
2026-06-16 09:23:23,839 - BasicTS-training - INFO - Building train data loader.
2026-06-16 09:23:23,842 - BasicTS-training - INFO - Set optim: Adam
2026-06-16 09:23:23,843 - BasicTS-training - INFO - Building val data loader.
2026-06-16 09:23:23,844 - BasicTS-training - INFO - Building test data loader.
2026-06-16 09:23:23,846 - BasicTS-training - INFO - Total parameters: 818188
2026-06-16 09:23:23,846 - BasicTS-training - INFO - Trainable parameters: 818188
2026-06-16 09:23:23,847 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [00:05<00:00, 52.65it/s]
2026-06-16 09:23:28,923 - BasicTS-training - INFO - Result <train>: [train/time: 5.07 (s), train/loss: 0.3395, train/MAE: 0.3395, train/MSE: 0.2523, train/RMSE: 0.5023, train/MAPE: 4.8815, train/WAPE: 0.6480]
2026-06-16 09:23:28,926 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 172.67it/s]
2026-06-16 09:23:29,

Benchmarking PatchTST...


100%|██████████| 267/267 [00:14<00:00, 18.59it/s]
2026-06-16 09:24:11,457 - BasicTS-training - INFO - Result <train>: [train/time: 14.37 (s), train/loss: 0.3536, train/MAE: 0.3536, train/MSE: 0.2760, train/RMSE: 0.5254, train/MAPE: 4.9939, train/WAPE: 0.6773]
2026-06-16 09:24:11,460 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:01<00:00, 63.81it/s]
2026-06-16 09:24:12,825 - BasicTS-training - INFO - Result <val>: [val/time: 1.36 (s), val/loss: 0.3936, val/MAE: 0.3936, val/MSE: 0.3521, val/RMSE: 0.5934, val/MAPE: 5.6932, val/WAPE: 0.6409]
2026-06-16 09:24:12,844 - BasicTS-training - INFO - Checkpoint checkpoints\benchmark\PatchTST\ETTh1_96_12\3e81bc4114ae9339f74cb900b5f999d3\PatchTSTForForecasting_best_val_MAE.pt saved
100%|██████████| 87/87 [00:01<00:00, 65.22it/s]
2026-06-16 09:24:14,182 - BasicTS-training - INFO - Result <test>: [test/time: 1.34 (s), test/loss: 0.3406, test/MAE: 0.3406, test/MSE: 0.2948, test/RMSE: 0.5430, test/MAPE: 8.9710, test/WAPE: 0.57

Benchmark finished. Results saved to results/


In [4]:
# Display final ranking and failures
import pandas as pd
results_csv = Path("results/model_benchmark_results.csv")
if results_csv.exists():
	df = pd.read_csv(results_csv)
	display(df.sort_values(["MAE", "MSE"], na_position="last"))
else:
	print("No successful model results found.")

failed_csv = Path("results/model_benchmark_failed.csv")
if failed_csv.exists():
	failed_df = pd.read_csv(failed_csv)
	print("\nFailed models:")
	display(failed_df)
else:
	print("No failures recorded.")


,model,MAE,MSE,RMSE,MAPE,WAPE,metrics_file,ckpt_dir
0,iTransformer,0.326405,0.272133,0.521664,8.672733,0.554847,checkpoints\benchmark\iTransformer\ETTh1_96_12...,checkpoints\benchmark\iTransformer\ETTh1_96_12
1,PatchTST,0.330091,0.280017,0.529167,9.006214,0.561137,checkpoints\benchmark\PatchTST\ETTh1_96_12\3e8...,checkpoints\benchmark\PatchTST\ETTh1_96_12
2,DLinear,0.331306,0.282716,0.531711,8.372759,0.557269,checkpoints\benchmark\DLinear\ETTh1_96_12\1e0a...,checkpoints\benchmark\DLinear\ETTh1_96_12
3,NLinear,0.335885,0.289058,0.537642,8.959574,0.568305,checkpoints\benchmark\NLinear\ETTh1_96_12\5484...,checkpoints\benchmark\NLinear\ETTh1_96_12
4,MTSMixer,0.366557,0.325800,0.570789,9.814173,0.605643,checkpoints\benchmark\MTSMixer\ETTh1_96_12\07a...,checkpoints\benchmark\MTSMixer\ETTh1_96_12



Failed models:


,model,error
0,TimesNet,train error: TimesNetForForecasting.forward() ...
